In [9]:
# ============================================================
# Arabic Summary Regeneration - Kaggle GPU Version (With Length Categories)
# ============================================================

# ── تثبيت المكتبات ─────────────────────────────

# ── استيراد المكتبات ───────────────────────────
import pandas as pd
import torch
import re
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from tqdm import tqdm
import os

# ── إعدادات الملفات ────────────────────────────
INPUT_CSV  = "summerize/results_csv/merged_datasetfinalv2 (2).csv"
OUTPUT_CSV = "summerize/results_csv/finaldataset.csv"
CHECKPOINT = "summerize/results_csv/remove/combined_dataset_checkpoint_TEST.csv"

# ── إعدادات النموذج ────────────────────────────
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"  # ← غيّر هنا
MAX_TOKENS = 300
TEXT_COL    = "text"
SUMMARY_COL = "summary"
TYPE_COL    = "type"

# 🧪 TEST MODE
TEST_MODE = True
TEST_ROWS = 9000

# ── دالة تصنيف الطول (بناءً على عدد الكلمات) ──
def get_length_category(word_count):
    if word_count < 100:
        return "short"
    elif word_count <= 300:
        return "medium"
    else:
        return "long"

# ── Prompts المحسّنة بدقة حسب المعايير المطلوبة ──
SYSTEM_PROMPT = """أنت متخصص في تلخيص النصوص العربية.
قواعد صارمة جداً:
1. اكتب الملخص باللغة العربية الفصحى فقط.
2. ممنوع تماماً استخدام أي لغة أخرى (إنجليزية، صينية، إلخ).
3. ممنوع كتابة عبارات مثل "ملخص النص" أو "الترجمة هي". ابدأ مباشرة بالمحتوى.
4. التزم بدقة بنسبة الطول والخصائص المطلوبة لكل فئة جمهور أدناه."""

PROMPTS = {
    "kids": """لخّص النص التالي لجمهور الأطفال.
المعايير الإلزامية:
- التعقيد: بسيط جداً.
- الطول: قصير جداً (يجب أن يكون حوالي 5% إلى 20% من طول النص الأصلي).
- التركيز: الفكرة الرئيسية فقط.
- الخصائص: استخدم مفردات بسيطة جداً يسهل على الطفل فهمها، وأنهِي الملخص بجملة إيجابية ومشجعة.
- ممنوع نسخ الجمل حرفياً من النص الأصلي.

النص الأصلي:
{text}""",

    "adults": """لخّص النص التالي للقارئ العام (البالغين).
المعايير الإلزامية:
- التعقيد: متوسط ومعتدل.
- الطول: متوسط (يجب أن يكون حوالي 10% إلى 30% من طول النص الأصلي).
- التركيز: الجوانب العملية والمفيدة.
- الخصائص: لغة واضحة وسهلة الوصول، وتجنب تماماً التفاصيل التقنية أو المعقدة أو المصطلحات المتخصصة.
- ممنوع نسخ الجمل حرفياً من النص الأصلي.

النص الأصلي:
{text}""",

    "experts": """لخّص النص التالي للمتخصصين والخبراء.
المعايير الإلزامية:
- التعقيد: تقني وعلمي.
- الطول: موجز ومكثف (يجب أن يكون حوالي 5% إلى 15% من طول النص الأصلي).
- التركيز: النتائج الدقيقة والمعلومات الجوهرية.
- الخصائص: استخدم المصطلحات العلمية والمجال التخصصي بدقة، بأسلوب أكاديمي رسمي وموجز جداً بدون أي حشو أو مقدمات.
- ممنوع نسخ الجمل حرفياً من النص الأصلي.

النص الأصلي:
{text}"""
}

# ── تحميل النموذج على GPU ─────────────────────
print("🤖 Loading model on GPU...")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    torch_dtype=torch.float16,
)
model.eval()

print(f"✅ Model loaded on: {next(model.parameters()).device}")

# ── دالة توليد الملخص (مع مصفاة الأمان) ────────
def generate_summary(text, audience):
    if audience not in PROMPTS:
        audience = "adults"
    
    user_prompt = PROMPTS[audience].format(text=str(text).strip()[:3000])
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt}
    ]
    
    text_input = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text_input, return_tensors="pt").to(model.device)
    
    for attempt in range(2):
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=MAX_TOKENS,
                temperature=0.1, # منخفض لضمان الالتزام الصارم بالتعليمات
                top_p=0.9,
                do_sample=True,
                pad_token_id=tokenizer.eos_token_id,
            )
        
        new_tokens = outputs[0][inputs['input_ids'].shape[1]:]
        summary = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        
        # 🛡️ مصفاة الأمان: منع الأحرف غير العربية أو عبارات الترجمة
        has_chinese = bool(re.search(r'[\u4e00-\u9fff]', summary))
        has_translation_note = "ترجمة" in summary or "ترجم" in summary or "بالصينية" in summary or "Summary" in summary
        
        if has_chinese or has_translation_note:
            print(f"    ⚠️  Detected invalid output. Retrying...")
            continue 
            
        return summary
    
    return ""

# ── كشف النسخ ─────────────────────────────────
def is_copy(text, summary):
    text_words = set(str(text).split())
    summary_words = set(str(summary).split())
    if not summary_words:
        return True
    overlap = len(summary_words & text_words) / len(summary_words)
    ratio = len(str(summary)) / max(len(str(text)), 1)
    return (overlap > 0.85 and ratio > 0.15) or ratio > 0.40

# ── تحميل البيانات ────────────────────────────
print("\n📂 Loading dataset...")
df = pd.read_csv(INPUT_CSV)
print(f"  Total rows: {len(df)}")

if TYPE_COL not in df.columns:
    raise ValueError(f"❌ Column '{TYPE_COL}' not found in CSV!")

if "_done" not in df.columns:
    df["_done"] = False

# ── اكتشاف الملخصات المنسوخة ─────────────────
print("\n🔍 Detecting copy-paste summaries...")
df["_is_copy"] = [
    False if row["_done"] else is_copy(row[TEXT_COL], row[SUMMARY_COL])
    for _, row in df.iterrows()
]
indices = df[df["_is_copy"]].index.tolist()

if TEST_MODE:
    indices = indices[:TEST_ROWS]
    print(f"  🧪 TEST MODE ACTIVE: Processing only the first {TEST_ROWS} rows.")
else:
    print(f"  Rows to regenerate: {len(indices)}")

# ── المعالجة ──────────────────────────────────
print("\n✍️  Generating summaries with GPU...\n")

audience_counts = {"kids": 0, "adults": 0, "experts": 0}
success = 0
failed = 0

for i, idx in enumerate(tqdm(indices, desc="Processing")):
    text = str(df.at[idx, TEXT_COL])
    audience = str(df.at[idx, TYPE_COL]).strip().lower()
    if audience not in PROMPTS:
        audience = "adults"
    audience_counts[audience] += 1
    
    if len(text.strip()) < 50:
        df.at[idx, "_done"] = True
        continue
    
    try:
        new_summary = generate_summary(text, audience)
        if new_summary:
            df.at[idx, SUMMARY_COL] = new_summary
            df.at[idx, "_done"] = True
            success += 1
            
            # طباعة المعلومات الجديدة للتقييم السريع
            orig_words = len(text.split())
            sum_words = len(new_summary.split())
            print(f"\n📝 Original ({orig_words} words, [{get_length_category(orig_words)}]): {text[:100]}...")
            print(f"✨ Summary  ({sum_words} words, [{get_length_category(sum_words)}]): {new_summary}\n" + "-"*70)
        else:
            failed += 1
    except Exception as e:
        print(f"\n❌ Error at row {idx}: {e}")
        failed += 1
        torch.cuda.empty_cache()
    
    if (i + 1) % 50 == 0:
        df.to_csv(CHECKPOINT, index=False, encoding="utf-8-sig")

# ── الحفظ النهائي وإضافة الأعمدة الجديدة ───────
print("\n💾 Calculating lengths and categories for final save...")

df["text_length_x"] = df[TEXT_COL].apply(lambda x: len(str(x).split()))
df["category_x"] = df["text_length_x"].apply(get_length_category)

df["text_length_y"] = df[SUMMARY_COL].apply(lambda x: len(str(x).split()))
df["category_y"] = df["text_length_y"].apply(get_length_category)

# إزالة الأعمدة المؤقتة
df.drop(columns=["_done", "_is_copy"], errors="ignore", inplace=True)

# الحفظ
df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")

if os.path.exists(CHECKPOINT):
    os.remove(CHECKPOINT)

print("\n" + "=" * 70)
print("✅ Test Done!")
print(f"  Regenerated: {success}")
print(f"  Failed     : {failed}")
print(f"  New columns added: text_length_x, category_x, text_length_y, category_y")
print(f"  Saved to: {OUTPUT_CSV}")
print("=" * 70)

🤖 Loading model on GPU...


ImportError: Using `bitsandbytes` 4-bit quantization requires the latest version of bitsandbytes: `pip install -U bitsandbytes`